# 6. NLP y modelos de lenguaje aplicados a finanzas
## tres formas de leer el sentimiento del mercado


### ¿Por qué nos importa el sentimiento de las noticias?

Un anuncio de "recorte de dividendos" o de "récord de utilidades"
puede mover un precio en minutos, ajustar la prima de riesgo que el mercado exige a una empresa, o alimentar un modelo de valoración que usa el sentimiento como una señal más, junto al PER o el ROE.

**Ningún analista** puede leer manualmente miles de titulares por día. Necesitamos automatizar la lectura.  

Existen varias formas de hacerlo, con compromisos muy distintos entre costo, velocidad, precisión e interpretabilidad.

### ¿Qué se pretende?

1. Explicar por qué el análisis de sentimiento en textos financieros es más difícil que en textos genéricos, y cómo la solución evolucionó de diccionarios a modelos de lenguaje.
2. Implementar y comparar **tres enfoques** de clasificación de sentimiento (diccionario financiero,
   FinBERT y un LLM generativo) sobre el **mismo conjunto de noticias financieras**.
3. Diseñar *prompts* estructurados (zero-shot y few-shot) para obtener salidas confiables en formato
   JSON desde un LLM.
4. Reconocer los riesgos de alucinación de los LLM en contextos financieros y aplicar buenas prácticas
   (extraer de documentos provistos, no preguntar "de memoria"; verificar siempre la salida).

Vamos a resolver **el mismo problema** clasificar una frase de noticia financiera como positiva,
negativa o neutral con tres técnicas cada vez más sofisticadas. Cada peldaño existe porque el
anterior tiene una limitación concreta que vamos a ver fallar con nuestros propios ojos:

| Peldaño | Técnica | Resuelve... | Pero falla cuando... |
|---|---|---|---|
| 1 | Diccionario financiero (Loughran-McDonald) | Palabras genéricas mal calificadas en finanzas | Hay negaciones o contexto ("no cayeron las utilidades") |
| 2 | Embeddings + FinBERT (encoder) | El contexto de cada palabra | Necesitamos que el modelo *explique* su razonamiento o siga instrucciones nuevas |
| 3 | LLM generativo con *prompting* (decoder) | Instrucciones flexibles y explicaciones en texto | El modelo puede "alucinar" datos que no están en el texto |

### 0.1 Entorno de ejecución

Este cuaderno está pensado para Colab:

- Las partes  1 y 2 funcionan bien en CPU (no necesitas GPU para esa parte).
- la parte 3 (LLM generativo) necesita **GPU** para correr en un tiempo razonable. En Colab:
  `Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU` (nivel gratuito).

Ejecuta las siguientes celdas en orden. Si Colab te pide reiniciar el entorno de ejecución después de
instalar librerías, hazlo antes de continuar.

In [ ]:
# Instalación de librerías con versiones fijadas para reproducibilidad.
#
# OJO: a propósito NO reinstalamos `torch` aquí. Colab ya trae una versión de
# PyTorch compilada específicamente contra la GPU del entorno (CUDA). Si la
# reemplazamos con `pip install torch==...` corremos el riesgo de instalar una
# build que no coincide con esa GPU y perder el soporte de GPU sin darnos
# cuenta. Todo lo demás sí lo fijamos.
#
# `bitsandbytes` solo publica wheels para Linux y Windows: en Colab (Linux)
# se instala sin problema. En MacOS no funciona.
!pip install -q \
    "transformers==4.46.3" \
    "datasets==3.0.1" \
    "accelerate==1.0.1" \
    "bitsandbytes==0.44.1" \
    "sentence-transformers==3.2.1" \
    "huggingface_hub==0.25.2" \
    "scikit-learn==1.5.2" \
    "pandas==2.2.3" \
    "matplotlib==3.9.2" \
    "seaborn==0.13.2"

In [ ]:
# Importaciones y configuración general
import os
import re
import json
import time
import random
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_colwidth", 120)

print(f"PyTorch: {torch.__version__}")

In [ ]:
# Detección de GPU
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = "cpu"
    print("⚠️  No se detectó GPU.")
    print("   Las partes 1 y 2 (diccionario y FinBERT) funcionan bien en CPU.")
    print("   la parte3 (LLM generativo) será MUY lento en CPU.")
    print("   En Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4).")

In [ ]:
# Semillas para reproducibilidad
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

<a id="sec1"></a>
## 1. Datos: Financial PhraseBank

El [**Financial PhraseBank**](https://huggingface.co/datasets/takala/financial_phrasebank) (Malo et al.,
2014) es un conjunto de ~4 840 frases tomadas de noticias financieras en inglés (Reuters, LexisNexis),
anotadas por 16 personas con formación en finanzas. Cada anotador clasificó cada frase como
**positiva, negativa o neutral**, siempre desde el punto de vista de un inversionista (¿esta frase
sugiere que el precio de la acción debería subir, bajar, o no debería verse afectado?).

El dataset viene en cuatro versiones según qué tan de acuerdo estuvieron los anotadores entre sí:

| Configuración | % de acuerdo exigido | Nº de frases (aprox.) |
|---|---|---|
| `sentences_allagree` | 100% de los anotadores | ~2 264 |
| `sentences_75agree` | ≥75% | ~3 453 |
| `sentences_66agree` | ≥66% | ~4 217 |
| `sentences_50agree` | ≥50% | ~4 846 |

Usaremos `sentences_allagree`: es la más pequeña, pero también la más confiable (nadie discutió la
etiqueta), lo cual la hace ideal para comparar métodos de forma justa.

### ⚠️ Un problema real: el dataset "no carga"

Si buscas ejemplos de este dataset en internet, casi todos usan:

```python
from datasets import load_dataset
load_dataset("takala/financial_phrasebank", "sentences_allagree")
```

Esto **ya no funciona** con versiones recientes de la librería `datasets`. El dataset se publicó en
Hugging Face usando un *script* de carga (`financial_phrasebank.py`) que descarga y descomprime el
archivo original. Por razones de seguridad, Hugging Face **dejó de permitir la ejecución de scripts de
carga arbitrarios**, así que `datasets` ahora rechaza este tipo de dataset directamente con un error
`RuntimeError: Dataset scripts are no longer supported`. Para completar el cuadro, el script además
apunta a un identificador de repositorio (`financial_phrasebank`, sin el prefijo `takala/`) que ya no
existe tal cual, así que ni siquiera forzando `trust_remote_code=True` funciona.

Esto es exactamente el tipo de fricción que te vas a encontrar en proyectos reales con datasets de
Hugging Face que llevan varios años publicados: **el archivo de datos original sigue estando ahí**
(alojado como un `.zip` normal en el repositorio), pero el mecanismo automático para leerlo se rompió.
La solución: descargarlo directamente y parsearlo nosotros mismos.

Abajo implementamos **plan A** (el camino "oficial", que dejamos por transparencia aunque hoy falle) y
un **plan B** que sí funciona: descarga directa del `.zip` oficial desde el repositorio del dataset.

In [ ]:
# Plan A: la forma "de libro de texto" de cargar el dataset.
# La dejamos aquí a propósito para que veas el error real y por qué hace falta un plan B.
from datasets import load_dataset

FPB_CONFIG = "sentences_allagree"  # otras opciones: sentences_75agree, sentences_66agree, sentences_50agree

df_fpb = None
try:
    fpb_dataset = load_dataset("takala/financial_phrasebank", FPB_CONFIG, trust_remote_code=True)
    df_fpb = fpb_dataset["train"].to_pandas().rename(columns={"sentence": "frase", "label": "etiqueta"})
    # Si `datasets` cargó el dataset, la etiqueta viene como entero (ClassLabel);
    # la convertimos a texto con el mismo mapeo definido en el dataset.
    class_label = fpb_dataset["train"].features["label"]
    df_fpb["etiqueta"] = df_fpb["etiqueta"].apply(class_label.int2str)
    print("Plan A funcionó: dataset cargado con `datasets.load_dataset`.")
except Exception as error:
    print("Plan A falló (es lo esperado hoy):")
    print(f"  {type(error).__name__}: {str(error)[:200]}")

In [ ]:
# Plan B: descarga directa del archivo oficial del dataset (mismos datos, sin script).
# El archivo sigue alojado públicamente en el repositorio; solo lo leemos nosotros mismos.
if df_fpb is None:
    print("Usando plan B: descarga directa del .zip oficial.")

    from huggingface_hub import hf_hub_download

    ruta_zip = hf_hub_download(
        repo_id="takala/financial_phrasebank",
        repo_type="dataset",
        filename="data/FinancialPhraseBank-v1.0.zip",
    )

    nombre_archivo_por_config = {
        "sentences_allagree": "Sentences_AllAgree.txt",
        "sentences_75agree": "Sentences_75Agree.txt",
        "sentences_66agree": "Sentences_66Agree.txt",
        "sentences_50agree": "Sentences_50Agree.txt",
    }
    nombre_archivo = nombre_archivo_por_config[FPB_CONFIG]

    filas = []
    with zipfile.ZipFile(ruta_zip) as z:
        ruta_interna = f"FinancialPhraseBank-v1.0/{nombre_archivo}"
        with z.open(ruta_interna) as f:
            # El archivo original está codificado en latin-1 (ISO-8859-1), no en UTF-8.
            # Si lo lees como UTF-8 vas a obtener un UnicodeDecodeError.
            texto = f.read().decode("latin-1")
        for linea in texto.splitlines():
            if "@" not in linea:
                continue
            frase, etiqueta = linea.rsplit("@", 1)
            filas.append({"frase": frase.strip(), "etiqueta": etiqueta.strip()})

    df_fpb = pd.DataFrame(filas)
    print(f"Plan B funcionó: {len(df_fpb)} frases cargadas directamente del archivo oficial.")

df_fpb = df_fpb.reset_index(drop=True)
df_fpb.head(3)

### 1.1 Exploración del dataset

In [ ]:
ETIQUETAS = ["negative", "neutral", "positive"]

conteo = df_fpb["etiqueta"].value_counts().reindex(ETIQUETAS)
print(conteo)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=conteo.index, y=conteo.values, order=ETIQUETAS, ax=ax)
ax.set_title(f"Distribución de clases — Financial PhraseBank ({FPB_CONFIG})")
ax.set_xlabel("Sentimiento")
ax.set_ylabel("Número de frases")
plt.show()

In [ ]:
# Cinco ejemplos de cada clase
for etiqueta in ETIQUETAS:
    print(f"\n=== {etiqueta.upper()} ===")
    ejemplos = df_fpb.loc[df_fpb["etiqueta"] == etiqueta, "frase"].sample(5, random_state=SEED)
    for frase in ejemplos:
        print(f"- {frase}")

### 1.2 Muestra de evaluación común

Para que la comparación entre los tres métodos sea **justa**, vamos a evaluarlos siempre sobre la
**misma muestra**. También reservamos unas pocas frases fuera de esa muestra para usarlas como ejemplos
de referencia (*few-shot*) de la parte 3, sin que el LLM "vea" antes las frases que luego vamos a
usar para evaluarlo.

`N_EVAL` queda como parámetro editable: bájalo si quieres iterar más rápido (por ejemplo mientras
depuras el prompt del LLM) y súbelo para la comparación final.

In [ ]:
N_EVAL = 300  # tamaño de la muestra de evaluación común a los tres métodos (ajústalo si lo necesitas)

indices_eval, _ = train_test_split(
    df_fpb.index,
    train_size=min(N_EVAL, len(df_fpb)),
    stratify=df_fpb["etiqueta"],
    random_state=SEED,
)

df_eval = df_fpb.loc[indices_eval].reset_index(drop=True)
df_resto = df_fpb.drop(index=indices_eval)

print(f"Muestra de evaluación: {len(df_eval)} frases")
print(df_eval["etiqueta"].value_counts(normalize=True).reindex(ETIQUETAS).round(3))

In [ ]:
# Un ejemplo por clase, tomado del resto (fuera de la muestra de evaluación),
# para usar como demostración few-shot en la parte 3.
ejemplos_few_shot = (
    df_resto.groupby("etiqueta", group_keys=False)[["frase", "etiqueta"]]
    .apply(lambda g: g.sample(1, random_state=SEED))
    .reset_index(drop=True)
)
ejemplos_few_shot

<a id="sec2"></a>
## 2. Parte 1: Diccionario financiero (Loughran-McDonald)

### 2.1 La idea: "bolsa de palabras"

El enfoque más simple para medir sentimiento es contar palabras: si una frase tiene más palabras de una
lista de "palabras positivas" que de una lista de "palabras negativas", la llamamos positiva, y
viceversa. A esto se le llama enfoque de **bolsa de palabras** (*bag of words*): no importa el orden de
las palabras ni cómo se relacionan entre sí, solo cuáles aparecen.

El primer problema es que **los diccionarios de sentimiento genéricos (pensados para reseñas de
productos o redes sociales) fallan sistemáticamente en finanzas**, porque el vocabulario financiero
tiene connotaciones propias. Por ejemplo:

- **"liability" (pasivo)** suena negativo en el uso cotidiano, pero es un término contable neutral:
  toda empresa tiene pasivos.
- **"tax" (impuesto)** suena negativo para una persona, pero en un reporte financiero puede aparecer en
  contextos completamente neutrales ("tax expense of $2M").
- **"cost" (costo)** no es necesariamente negativo: "cost reduction" (reducción de costos) es una
  noticia positiva para el inversionista, aunque contenga la palabra "cost".

Por eso, Loughran y McDonald (2011) construyeron un diccionario de sentimiento **específico para
textos financieros** (reportes 10-K de la SEC), donde palabras como "liability" simplemente no
aparecen en la lista negativa, y sí aparecen palabras propias del lenguaje de riesgo financiero
("litigation", "impairment", "default", "bankruptcy"...).

### 2.2 Obtener el diccionario

El diccionario Loughran-McDonald se distribuye desde la página de la Universidad de Notre Dame:
[sraf.nd.edu/loughranmcdonald-master-dictionary](https://sraf.nd.edu/loughranmcdonald-master-dictionary/)
(sección "Master Dictionary"). No se puede descargar mediante código porque hay que aceptar los
términos de uso en un formulario, así que aquí te damos dos caminos:

- **(a) Si ya descargaste el CSV oficial**: súbelo cuando la celda te lo pida (el nombre del archivo
  suele ser algo como `Loughran-McDonald_MasterDictionary_1993-2024.csv`).
- **(b) Si no lo tienes a mano**: seguimos con una **versión simplificada**, embebida en el código, con
  ~50 palabras positivas y ~50 negativas representativas del diccionario real. **No es el diccionario
  completo** (la versión oficial tiene cerca de dos mil palabras negativas y varios cientos de
  positivas) — es suficiente para ver el enfoque funcionar y, sobre todo, para ver dónde falla.

In [ ]:
# (a) Intento de carga del CSV oficial (solo tiene efecto si lo subes; si cancelas, seguimos con (b))
usar_csv_oficial = False
palabras_positivas, palabras_negativas = set(), set()

try:
    from google.colab import files

    print("Si tienes el CSV oficial de Loughran-McDonald, súbelo ahora.")
    print("Si prefieres usar la versión simplificada, presiona 'Cancel upload'.")
    archivos_subidos = files.upload()
except (ImportError, ModuleNotFoundError):
    archivos_subidos = {}
    print("No estamos en Colab: seguimos directo con la versión simplificada (b).")

if archivos_subidos:
    ruta_csv = list(archivos_subidos.keys())[0]
    df_lm = pd.read_csv(ruta_csv, encoding="latin-1")
    df_lm.columns = [c.strip().lower() for c in df_lm.columns]
    palabras_positivas = set(df_lm.loc[df_lm["positive"].astype(float) > 0, "word"].str.lower())
    palabras_negativas = set(df_lm.loc[df_lm["negative"].astype(float) > 0, "word"].str.lower())
    usar_csv_oficial = True
    print(f"CSV oficial cargado: {len(palabras_positivas)} palabras positivas, "
          f"{len(palabras_negativas)} negativas.")

In [ ]:
# (b) Versión simplificada con fines didácticos: ~50 palabras representativas por clase,
# tomadas del espíritu del diccionario Loughran-McDonald. NO reemplaza al diccionario oficial.
PALABRAS_POSITIVAS_SIMPLIFICADO = {
    "improve", "improved", "improvement", "improving", "improves",
    "growth", "grew", "growing", "grows",
    "gain", "gains", "gained",
    "profit", "profits", "profitable", "profitability",
    "strong", "strength", "strengthen", "strengthened",
    "success", "successful", "successfully",
    "achieve", "achieved", "achievement",
    "exceed", "exceeded", "exceeding",
    "outperform", "outperformed",
    "efficient", "efficiency",
    "innovative", "innovation",
    "favorable", "favorably",
    "opportunity", "opportunities",
    "robust", "solid",
    "upturn", "expansion", "expand", "expanded",
    "rebound", "recovery", "recovered",
    "enhance", "enhanced", "enhancement",
    "outstanding", "exceptional",
}

PALABRAS_NEGATIVAS_SIMPLIFICADO = {
    "loss", "losses",
    "decline", "declined", "declining", "declines",
    "deficit",
    "deteriorate", "deteriorated", "deterioration",
    "adverse", "adversely",
    "bankruptcy", "bankrupt",
    "impairment", "impair", "impaired",
    "litigation", "lawsuit", "lawsuits",
    "penalty", "penalties",
    "fraud", "fraudulent",
    "default", "defaults", "defaulted",
    "restructuring",
    "layoff", "layoffs",
    "downturn", "recession",
    "volatility", "volatile",
    "weak", "weakness", "weakened",
    "unfavorable", "negative",
    "failure", "failed", "fail",
    "breach", "violation", "violations",
    "terminate", "terminated", "termination",
    "delinquent", "delinquency",
    "shortfall", "closure", "closures",
    "discontinued", "insolvent", "insolvency",
}

if not usar_csv_oficial:
    palabras_positivas = PALABRAS_POSITIVAS_SIMPLIFICADO
    palabras_negativas = PALABRAS_NEGATIVAS_SIMPLIFICADO
    print(f"Usando versión simplificada: {len(palabras_positivas)} palabras positivas, "
          f"{len(palabras_negativas)} negativas.")

### 2.3 Clasificador por conteo de palabras

La regla es simple: tokenizamos la frase (minúsculas, solo palabras), contamos cuántos tokens caen en
cada lista, y decidimos por mayoría. Empate → neutral.

In [ ]:
def tokenizar(texto):
    """Tokenización simple: minúsculas y solo secuencias de letras."""
    return re.findall(r"[a-záéíóúñ]+", texto.lower())


def clasificar_por_diccionario(texto, positivas, negativas):
    tokens = tokenizar(texto)
    n_pos = sum(1 for t in tokens if t in positivas)
    n_neg = sum(1 for t in tokens if t in negativas)
    if n_pos > n_neg:
        return "positive"
    if n_neg > n_pos:
        return "negative"
    return "neutral"


tiempos = {}
resultados = []

inicio = time.perf_counter()
df_eval["pred_diccionario"] = df_eval["frase"].apply(
    lambda t: clasificar_por_diccionario(t, palabras_positivas, palabras_negativas)
)
tiempos["Diccionario"] = (time.perf_counter() - inicio) / len(df_eval)

### 2.4 Evaluación

Vamos a reutilizar estas dos funciones de evaluación para los **cuatro** métodos que comparemos en
este cuaderno (diccionario, FinBERT, LLM zero-shot, LLM few-shot), para que las métricas sean
directamente comparables.

In [ ]:
def evaluar(y_true, y_pred, nombre_metodo):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, labels=ETIQUETAS, average="macro")
    print(f"--- {nombre_metodo} ---")
    print(f"Accuracy: {acc:.3f}  |  F1 macro: {f1_macro:.3f}")
    print(classification_report(y_true, y_pred, labels=ETIQUETAS, digits=3, zero_division=0))
    return {"metodo": nombre_metodo, "accuracy": acc, "f1_macro": f1_macro}


def graficar_matriz_confusion(y_true, y_pred, nombre_metodo, ax=None):
    cm = confusion_matrix(y_true, y_pred, labels=ETIQUETAS)
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=ETIQUETAS, yticklabels=ETIQUETAS, ax=ax, cbar=False,
    )
    ax.set_title(nombre_metodo)
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    return ax

In [ ]:
resultados.append(evaluar(df_eval["etiqueta"], df_eval["pred_diccionario"], "Diccionario"))
graficar_matriz_confusion(df_eval["etiqueta"], df_eval["pred_diccionario"], "Diccionario")
plt.show()

### 2.5 Limitación visible: el diccionario no entiende negaciones

El conteo de palabras no tiene forma de saber que "not" invierte el sentido de la palabra que sigue.
Veamos qué pasa con frases que contienen negaciones explícitas:

In [ ]:
ejemplos_negacion = [
    "Profits did not decline in the third quarter.",
    "The company avoided losses despite the difficult market conditions.",
    "Sales did not fail to meet analyst expectations.",
]

for frase in ejemplos_negacion:
    pred = clasificar_por_diccionario(frase, palabras_positivas, palabras_negativas)
    tokens_detectados = [t for t in tokenizar(frase) if t in palabras_positivas or t in palabras_negativas]
    print(f"Frase: {frase}")
    print(f"  Palabras del diccionario detectadas: {tokens_detectados}")
    print(f"  Predicción: {pred}  (la frase en realidad es neutral o positiva para el inversionista)\n")

Fíjate en el resultado de la primera frase: el diccionario la clasifica como **neutral**, que a
simple vista parece razonable... pero llegó ahí **por la razón equivocada**. Detectó *profits*
(positiva) y *decline* (negativa), y como hubo empate, el desempate por defecto la mandó a neutral —
no porque "entendiera" que "did not decline" invierte el sentido de *decline*. Si la frase hubiera
sido "Operating losses did not decline" (dos palabras negativas, ninguna positiva), el mismo mecanismo
la habría marcado como negativa a pesar de describir algo malo que **no empeoró**.

Las otras dos frases sí fallan de forma más obvia: ambas describen algo bueno para la empresa (pérdidas
que se **evitaron**, ventas que **no** decepcionaron), pero el diccionario las marca como negativas
porque solo ve las palabras *losses* y *fail*, sin registrar que "avoided" y "did not fail" invierten
su sentido. El diccionario cuenta palabras sin mirar la estructura de la frase, así que se equivoca
sistemáticamente con negaciones, condicionales o comparaciones ("less bad than expected").

**Pregunta que motiva la siguiente parte:** ¿cómo logramos que el modelo entienda que el significado
de una palabra depende de las palabras que la rodean?

<a id="sec3"></a>
## 3. Parte 2: Embeddings, atención y FinBERT

### 3.1 Embeddings: palabras como puntos en un espacio

Un **embedding** es una representación numérica de una palabra (o frase) como un vector de números
reales, aprendido de tal forma que **palabras con significados parecidos quedan cerca en ese espacio**.

Si conoces el análisis factorial o los modelos multifactoriales en finanzas, la idea es análoga: en vez
de describir cada acción con decenas de variables observadas (P/E, ROE, beta, momentum...), un modelo
de factores la resume en unas pocas coordenadas latentes (tamaño, valor, momentum...) tales que acciones
parecidas quedan cerca en ese espacio de factores. Un embedding hace lo mismo con palabras: en lugar de
describir "profit" con reglas explícitas, un modelo de lenguaje le asigna un vector de, por ejemplo, 384
números, aprendidos de tal forma que "profit" quede cerca de "earnings" y lejos de "bankruptcy".

### 3.2 Atención: el significado depende del contexto

El problema de los embeddings *estáticos* (una palabra → un vector fijo) es que una misma palabra puede
significar cosas distintas según el contexto: "shares" puede ser "acciones" (financiero) o "comparte"
(verbo); "bank" puede ser un banco o la orilla de un río. El mecanismo de **atención**, introducido por
la arquitectura *Transformer*, resuelve esto: para cada palabra de la frase, el modelo calcula cuánto
debe "prestarle atención" a cada una de las demás palabras para ajustar su significado. Es como un
analista que, al leer "the company's shares rose after strong earnings", automáticamente interpreta
"shares" en su sentido financiero porque el resto de la frase (*earnings*, *rose*) le da ese contexto —
sin que nadie se lo tenga que indicar explícitamente.

### 3.3 BERT: un encoder hecho para entender

**BERT** (*Bidirectional Encoder Representations from Transformers*) es un *encoder*: lee la frase
completa de una vez (en ambas direcciones, de ahí "bidirectional") y produce una representación densa
de su significado. Está diseñado específicamente para tareas de **comprensión y clasificación** —no
para generar texto nuevo—, lo que lo hace rápido y eficiente para justo el tipo de tarea que tenemos
aquí: decidir si una frase es positiva, negativa o neutral.

**FinBERT** es una versión de BERT preentrenada adicionalmente sobre texto financiero y afinada
específicamente para clasificar sentimiento financiero — es decir, ya "sabe" que "liability" no es una
palabra negativa, porque aprendió esa asociación directamente de millones de textos financieros reales,
en lugar de depender de una lista de palabras que nosotros mismos armamos a mano.

### 3.4 Demostración: embeddings de palabras financieras

Antes de usar FinBERT para clasificar, veamos con nuestros propios ojos que los embeddings capturan
significado: tomamos ~15 palabras financieras, obtenemos su embedding con un modelo pequeño
(`all-MiniLM-L6-v2`) y las proyectamos a 2D con PCA para poder graficarlas.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

modelo_embeddings = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

palabras_finanzas = [
    "profit", "revenue", "earnings", "growth", "dividend",
    "loss", "debt", "decline", "bankruptcy", "deficit",
    "merger", "acquisition", "recession", "inflation", "investment",
]

embeddings_palabras = modelo_embeddings.encode(palabras_finanzas)
print(f"Forma de los embeddings: {embeddings_palabras.shape}")

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
coordenadas_2d = pca.fit_transform(embeddings_palabras)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(coordenadas_2d[:, 0], coordenadas_2d[:, 1])
for palabra, (x, y) in zip(palabras_finanzas, coordenadas_2d):
    ax.annotate(palabra, (x, y), textcoords="offset points", xytext=(5, 5))
ax.set_title("Embeddings de palabras financieras, reducidos a 2D con PCA")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
plt.show()

Fíjate en los grupos que se forman: palabras de resultados positivos (*profit*, *earnings*, *revenue*,
*growth*, *dividend*) tienden a quedar cerca entre sí, y separadas de las palabras de dificultad
financiera (*loss*, *debt*, *decline*, *bankruptcy*, *deficit*). Esto no lo programamos a mano — emergió
del entrenamiento del modelo sobre grandes cantidades de texto. **Nota:** al ser un modelo pequeño y
genérico (no especializado en finanzas), la separación no será perfecta; con FinBERT, que sí está
afinado para este dominio, la distinción es mucho más nítida.

### 3.5 Clasificación con FinBERT

Usamos `ProsusAI/finbert` a través de un `pipeline` de `transformers`, que se encarga de tokenizar,
pasar por el modelo y decodificar la predicción por nosotros.

In [ ]:
from transformers import pipeline

FINBERT_MODEL_NAME = "ProsusAI/finbert"

clasificador_finbert = pipeline(
    "text-classification",
    model=FINBERT_MODEL_NAME,
    device=0 if DEVICE == "cuda" else -1,
)

# Antes de confiar en las etiquetas que devuelve el pipeline, verificamos el
# mapeo id -> etiqueta que trae el propio modelo (esto puede variar entre
# checkpoints: algunos devuelven "LABEL_0"/"LABEL_1"/"LABEL_2" en vez de texto).
print(clasificador_finbert.model.config.id2label)

In [ ]:
# Inferencia por lotes sobre la misma muestra de evaluación
frases_eval = df_eval["frase"].tolist()

inicio = time.perf_counter()
salidas_finbert = clasificador_finbert(frases_eval, batch_size=32, truncation=True)
tiempos["FinBERT"] = (time.perf_counter() - inicio) / len(frases_eval)

# El id2label de ProsusAI/finbert ya devuelve texto ("positive"/"negative"/"neutral"),
# así que en principio no habría que traducir nada; dejamos el mapeo explícito de
# todas formas, como salvaguarda por si algún día cambia el checkpoint.
MAPA_FINBERT = {"positive": "positive", "negative": "negative", "neutral": "neutral"}
df_eval["pred_finbert"] = [MAPA_FINBERT[s["label"]] for s in salidas_finbert]

In [ ]:
resultados.append(evaluar(df_eval["etiqueta"], df_eval["pred_finbert"], "FinBERT"))
graficar_matriz_confusion(df_eval["etiqueta"], df_eval["pred_finbert"], "FinBERT")
plt.show()

### 3.6 Volvemos a los ejemplos de negación

¿FinBERT sí entiende el contexto de las frases donde el diccionario falló?

In [ ]:
salidas_negacion = clasificador_finbert(ejemplos_negacion, top_k=None)

for frase, salida in zip(ejemplos_negacion, salidas_negacion):
    probabilidades = {d["label"]: round(d["score"], 3) for d in salida}
    prediccion = max(salida, key=lambda d: d["score"])["label"]
    print(f"Frase: {frase}")
    print(f"  Predicción FinBERT: {prediccion}")
    print(f"  Probabilidades: {probabilidades}\n")

Compara estas predicciones con las del diccionario en la sección 2.5. FinBERT no cuenta palabras
sueltas: procesa la frase completa con atención, así que "did not decline" y "avoided losses" se leen
como unidades con un significado propio — no como una simple suma de palabras positivas y negativas.

**Pregunta que motiva la siguiente parte:** FinBERT clasifica muy bien, pero solo sabe hacer
*exactamente* la tarea para la que fue afinado (positivo/negativo/neutral) y no puede explicarnos su
razonamiento en palabras. ¿Qué hacemos si mañana queremos pedirle, sin reentrenar nada, que además
identifique el tema de la noticia, o que justifique su respuesta?

<a id="sec4"></a>
## 4. Parte 3: LLM generativo (modelo decoder) con *prompting*

### 4.1 De entender a generar

Los modelos de lenguaje generativos (GPT, Claude, Qwen, Llama...) usan la misma arquitectura
*Transformer* que BERT, pero con una diferencia clave: son **decoders**, entrenados con un objetivo
mucho más simple en apariencia — predecir la siguiente palabra de un texto, una y otra vez, sobre
cantidades enormes de texto. Lo sorprendente es que, a suficiente escala (parámetros, datos, cómputo),
ese objetivo tan simple da lugar a capacidades emergentes sofisticadas: razonamiento paso a paso,
seguimiento de instrucciones, generación de código. Con una etapa adicional de afinamiento por
instrucciones (*instruction tuning*), estos modelos aprenden además a comportarse como asistentes que
siguen órdenes en lenguaje natural, en vez de simplemente completar texto.

Esto abre dos formas de usarlos sin entrenar nada:

- **Zero-shot**: le damos únicamente la instrucción ("clasifica el sentimiento de esta frase...") y
  esperamos que la siga, sin ejemplos previos.
- **Few-shot**: además de la instrucción, le mostramos unos pocos ejemplos ya resueltos (frase →
  respuesta correcta) para que "entienda" mejor el formato y el criterio que esperamos.

**Analogía:** FinBERT es como un analista junior *hiperespecializado*, entrenado durante meses para
hacer exactamente una tarea (clasificar sentimiento financiero) y que la hace rápido y bien, pero no
sabe hacer nada más. Un LLM generativo es como un analista senior generalista, muy capaz, al que puedes
pedirle prácticamente cualquier tarea de lenguaje — pero que necesita que le des instrucciones claras
(un buen *prompt*), y cuyas respuestas conviene siempre revisar.

### 4.2 El modelo: Qwen2.5-3B-Instruct

Vamos a usar [`Qwen/Qwen2.5-3B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct), un modelo
de pesos abiertos y descarga libre en Hugging Face (no requiere solicitar aprobación de acceso).

> ⚠️ **Nota de licencia:** a diferencia de otros tamaños de la misma familia (0.5B, 1.5B y 7B, que sí
> son Apache 2.0), **Qwen2.5-3B-Instruct se distribuye bajo la "Qwen Research License"**, pensada para
> investigación y uso no comercial (revisa los términos exactos en su
> [página de licencia](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/blob/main/LICENSE) antes de
> usarlo fuera de un contexto académico). Si en tu proyecto necesitas una licencia permisiva tipo
> Apache 2.0 con un tamaño similar, `Qwen/Qwen2.5-7B-Instruct` es una alternativa igual de sencilla de
> cargar en una T4 con 4 bits.

Dejamos el nombre del modelo en una variable para que puedas cambiarlo fácilmente por otro (por ejemplo
`"Qwen/Qwen2.5-7B-Instruct"`, algún modelo de la familia Gemma, o Llama).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # cámbialo aquí para probar otro modelo

tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
if tokenizer_llm.pad_token is None:
    tokenizer_llm.pad_token = tokenizer_llm.eos_token
# Necesario para generar por lotes con un modelo decoder-only: si el relleno
# (padding) va a la derecha, el modelo puede terminar "leyendo" tokens de
# relleno como si fueran parte real del prompt.
tokenizer_llm.padding_side = "left"

if DEVICE == "cuda":
    memoria_gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Un modelo de 3B en float16 ocupa ~6GB: sobra en una T4 de 16GB. Dejamos
    # los 4 bits como respaldo automático solo si la GPU disponible es chica.
    usar_4bit = memoria_gpu_gb < 10
    if usar_4bit:
        print(f"GPU con {memoria_gpu_gb:.1f} GB: cargando en 4 bits con bitsandbytes.")
        config_cuantizacion = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
        )
        modelo_llm = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME, quantization_config=config_cuantizacion, device_map="auto",
        )
    else:
        print(f"GPU con {memoria_gpu_gb:.1f} GB: cargando en float16 (sin necesidad de 4 bits).")
        modelo_llm = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
        )
else:
    print("Sin GPU: cargando en CPU en precisión float32. Esto será lento;")
    print("considera reducir N_EVAL antes de continuar si estás en CPU.")
    modelo_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_NAME)

modelo_llm.eval()

# El generation_config por defecto de muchos modelos instruct trae parámetros
# de muestreo (temperature, top_p) pensados para generación creativa. Como
# vamos a generar de forma determinística (do_sample=False), los desactivamos
# explícitamente aquí para no verlos ni tener que repetirlos en cada llamada.
modelo_llm.generation_config.do_sample = False
modelo_llm.generation_config.temperature = None
modelo_llm.generation_config.top_p = None
modelo_llm.generation_config.top_k = None

print(f"Modelo cargado: {LLM_MODEL_NAME}")

### 4.3 Diseño del prompt: forzar una salida en JSON

Le pedimos al modelo que responda **solo** con un objeto JSON con dos claves: `sentimiento` (una de
`positive`, `negative`, `neutral` — las mismas etiquetas que usamos en todo el cuaderno, para poder
comparar directamente) y `justificacion` (una frase breve en español). Usamos el *chat template* propio
del tokenizer, que da formato al prompt exactamente como el modelo espera verlo, y generación
**determinística** (`do_sample=False`): para una tarea de clasificación no queremos aleatoriedad en la
salida, sino la respuesta más probable según el modelo.

In [ ]:
PROMPT_SISTEMA = (
    "Eres un analista financiero. Vas a clasificar el sentimiento de una frase de "
    "una noticia financiera desde el punto de vista de un inversionista.\n"
    "Responde ÚNICAMENTE con un objeto JSON válido, sin texto adicional antes ni "
    "después, con exactamente estas dos claves:\n"
    '  "sentimiento": una de "positive", "negative" o "neutral"\n'
    '  "justificacion": una frase breve en español explicando tu decisión\n'
    "No agregues explicaciones fuera del JSON ni uses bloques de código markdown."
)


def construir_mensajes(frase, ejemplos_few_shot=None):
    """Arma la lista de mensajes (system/user/assistant) para el chat template."""
    mensajes = [{"role": "system", "content": PROMPT_SISTEMA}]
    if ejemplos_few_shot is not None:
        for _, fila in ejemplos_few_shot.iterrows():
            mensajes.append({"role": "user", "content": fila["frase"]})
            respuesta_ejemplo = json.dumps(
                {"sentimiento": fila["etiqueta"], "justificacion": "Ejemplo de referencia."},
                ensure_ascii=False,
            )
            mensajes.append({"role": "assistant", "content": respuesta_ejemplo})
    mensajes.append({"role": "user", "content": frase})
    return mensajes


# Vista rápida del prompt que efectivamente recibe el modelo (zero-shot)
prompt_ejemplo = tokenizer_llm.apply_chat_template(
    construir_mensajes(df_eval["frase"].iloc[0]), tokenize=False, add_generation_prompt=True,
)
print(prompt_ejemplo)

### 4.4 Parser robusto de la salida

En la práctica, un LLM casi nunca devuelve *exactamente* el JSON pedido: a veces agrega texto antes o
después, comillas raras, o simplemente no logra producir JSON válido. Escribimos un parser tolerante
que busca el primer bloque `{...}` en la respuesta y, si algo falla, **degrada a "neutral"** y marca la
fila como error de formato en vez de romper todo el pipeline.

In [ ]:
def parsear_respuesta_llm(texto):
    """Extrae {'sentimiento': ..., 'justificacion': ...} de la respuesta del LLM.

    Si no se puede extraer un sentimiento válido, devuelve ('neutral', '', True),
    donde el último valor indica que hubo un error de formato.
    """
    coincidencia = re.search(r"\{.*?\}", texto, re.DOTALL)
    if coincidencia:
        try:
            datos = json.loads(coincidencia.group())
            sentimiento = str(datos.get("sentimiento", "")).strip().lower()
            if sentimiento in ETIQUETAS:
                return sentimiento, str(datos.get("justificacion", "")), False
        except json.JSONDecodeError:
            pass
    return "neutral", "", True

### 4.5 Generación por lotes (zero-shot y few-shot)

Definimos una función de generación que procesa las frases en lotes (más eficiente que una por una),
con una barra de progreso y medición del tiempo total.

In [ ]:
from tqdm.auto import tqdm


def generar_respuestas(frases, ejemplos_few_shot=None, tam_lote=8, max_new_tokens=100):
    """Genera una respuesta del LLM por cada frase, en lotes. Devuelve (respuestas, tiempo_total)."""
    respuestas = []
    inicio = time.perf_counter()
    for i in tqdm(range(0, len(frases), tam_lote), desc="Generando"):
        lote = frases[i : i + tam_lote]
        prompts = [
            tokenizer_llm.apply_chat_template(
                construir_mensajes(f, ejemplos_few_shot), tokenize=False, add_generation_prompt=True,
            )
            for f in lote
        ]
        entradas = tokenizer_llm(prompts, return_tensors="pt", padding=True).to(modelo_llm.device)
        with torch.no_grad():
            salidas = modelo_llm.generate(
                **entradas,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer_llm.pad_token_id,
            )
        for entrada_ids, salida_ids in zip(entradas["input_ids"], salidas):
            texto_generado = tokenizer_llm.decode(
                salida_ids[len(entrada_ids):], skip_special_tokens=True,
            )
            respuestas.append(texto_generado)
    tiempo_total = time.perf_counter() - inicio
    return respuestas, tiempo_total

In [ ]:
# --- Zero-shot ---
respuestas_zero, tiempo_zero = generar_respuestas(df_eval["frase"].tolist(), ejemplos_few_shot=None)

parseo_zero = [parsear_respuesta_llm(r) for r in respuestas_zero]
df_eval["pred_llm_zero"] = [p[0] for p in parseo_zero]
df_eval["justificacion_llm_zero"] = [p[1] for p in parseo_zero]
df_eval["error_formato_zero"] = [p[2] for p in parseo_zero]
tiempos["LLM zero-shot"] = tiempo_zero / len(df_eval)

print(f"Tiempo total zero-shot: {tiempo_zero:.1f} s ({tiempos['LLM zero-shot']*1000:.1f} ms/frase)")
print(f"Respuestas no parseables: {df_eval['error_formato_zero'].sum()} de {len(df_eval)}")

In [ ]:
# --- Few-shot (3 ejemplos, uno por clase, tomados fuera de la muestra de evaluación) ---
respuestas_few, tiempo_few = generar_respuestas(
    df_eval["frase"].tolist(), ejemplos_few_shot=ejemplos_few_shot,
)

parseo_few = [parsear_respuesta_llm(r) for r in respuestas_few]
df_eval["pred_llm_few"] = [p[0] for p in parseo_few]
df_eval["justificacion_llm_few"] = [p[1] for p in parseo_few]
df_eval["error_formato_few"] = [p[2] for p in parseo_few]
tiempos["LLM few-shot"] = tiempo_few / len(df_eval)

print(f"Tiempo total few-shot: {tiempo_few:.1f} s ({tiempos['LLM few-shot']*1000:.1f} ms/frase)")
print(f"Respuestas no parseables: {df_eval['error_formato_few'].sum()} de {len(df_eval)}")

In [ ]:
resultados.append(evaluar(df_eval["etiqueta"], df_eval["pred_llm_zero"], "LLM zero-shot"))
resultados.append(evaluar(df_eval["etiqueta"], df_eval["pred_llm_few"], "LLM few-shot"))

### 4.6 Cinco ejemplos con la justificación del modelo

In [ ]:
df_eval.sample(5, random_state=SEED)[
    ["frase", "etiqueta", "pred_llm_few", "justificacion_llm_few"]
]

<a id="sec5"></a>
## 5. Demostración de riesgos: alucinaciones

Un LLM genera la continuación de texto **más probable** dado su entrenamiento — no consulta una base
de datos para verificar hechos. Cuando le preguntas algo que no está en su entrenamiento (o que
requiere datos actualizados que no tiene), puede generar una respuesta **con total seguridad** que
simplemente es incorrecta. A esto se le llama **alucinación**, y es uno de los riesgos más importantes
de usar LLMs en finanzas, donde una cifra inventada puede parecer perfectamente creíble.

Vamos a comparar dos formas de preguntar lo mismo: pedirle al modelo una cifra **"de memoria"**, contra
pedirle que la **extraiga de un texto que le damos**.

In [ ]:
def generar_respuesta_libre(mensajes, max_new_tokens=150):
    """Genera texto libre (sin forzar formato JSON) para una sola lista de mensajes."""
    prompt = tokenizer_llm.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer_llm(prompt, return_tensors="pt").to(modelo_llm.device)
    with torch.no_grad():
        salida = modelo_llm.generate(
            **entradas,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer_llm.pad_token_id,
        )
    return tokenizer_llm.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

### 5.1 Preguntar "de memoria"

Le pedimos una cifra trimestral específica de una empresa real, sin darle ningún dato en el prompt.

In [ ]:
pregunta_de_memoria = [
    {
        "role": "user",
        "content": (
            "¿Cuál fue el EBITDA de Grupo Argos en el tercer trimestre de 2023? "
            "Responde con la cifra exacta en pesos colombianos."
        ),
    }
]

respuesta_memoria = generar_respuesta_libre(pregunta_de_memoria)
print(respuesta_memoria)

Presta atención al **tono** de la respuesta, no solo al número: si el modelo responde con una cifra
concreta y un tono seguro, sin advertir que no tiene forma de verificarla, esa confianza aparente es
justamente el problema — no hay manera de distinguir, solo por cómo está redactada la respuesta, una
cifra real de una inventada. Antes de usar cualquier cifra que un LLM te dé "de memoria" en un contexto
financiero, **debes verificarla contra una fuente primaria** (el reporte financiero real de la empresa).

### 5.2 Extraer de un texto dado

Ahora le damos un párrafo **corto, corto y explícitamente marcado como ficticio**, y le pedimos que
extraiga el mismo tipo de dato — pero esta vez el dato sí está disponible en el contexto.

In [ ]:
contexto_ficticio = (
    "[Dato de ejemplo, inventado únicamente para esta demostración; no corresponde a ninguna "
    "empresa real]\n"
    "Según el reporte trimestral de la empresa ACME S.A., el EBITDA del tercer trimestre de 2023 "
    "fue de COP 128 400 millones, un 6% más que en el trimestre anterior."
)

pregunta_extraccion = [
    {
        "role": "user",
        "content": (
            f"{contexto_ficticio}\n\n"
            "¿Cuál fue el EBITDA de ACME S.A. en el tercer trimestre de 2023, según el texto anterior? "
            "Responde solo con la cifra."
        ),
    }
]

respuesta_extraccion = generar_respuesta_libre(pregunta_extraccion, max_new_tokens=60)
print(respuesta_extraccion)

Esta vez el modelo no tiene que "adivinar" nada: el número está literalmente en el texto que le dimos,
así que su tarea es de **extracción**, no de recuperación de memoria. Esta es la diferencia que importa
en la práctica: cuando le preguntas a un LLM algo "de memoria", estás confiando en que haya memorizado
correctamente un dato específico entre billones de tokens de entrenamiento (y que ese dato siga
vigente); cuando le pides que extraiga o resuma algo **de un documento que tú le proporcionas**, la
tarea es mucho más confiable, y el modelo tiene mucho menos margen para inventar.

**La lección para finanzas:** un LLM en un flujo de trabajo financiero debe trabajar sobre documentos
que tú le entregas (reportes, transcripciones, noticias) — nunca como fuente de verdad de memoria — y
toda salida que produzca (una cifra, una clasificación, un resumen) debe poder verificarse contra la
fuente original antes de usarse para una decisión real.

<a id="sec6"></a>
## 6. Comparación final

Reunimos las métricas de los cuatro enfoques evaluados sobre la misma muestra (`df_eval`), junto con
información cualitativa sobre cada uno (necesidad de GPU, si explica su decisión, costo aproximado).

In [ ]:
metadatos_metodos = {
    "Diccionario": {
        "requiere_gpu": "No",
        "explica_decision": "Sí (palabras contadas)",
        "costo_aprox": "Nulo (reglas en Python)",
    },
    "FinBERT": {
        "requiere_gpu": "No (recomendado si hay)",
        "explica_decision": "Parcial (probabilidades, sin texto)",
        "costo_aprox": "Bajo (modelo propio ~440 MB, sin API)",
    },
    "LLM zero-shot": {
        "requiere_gpu": "Sí (o CPU muy lento)",
        "explica_decision": "Sí (justificación en texto)",
        "costo_aprox": "Medio-alto (más cómputo por frase)",
    },
    "LLM few-shot": {
        "requiere_gpu": "Sí (o CPU muy lento)",
        "explica_decision": "Sí (justificación en texto)",
        "costo_aprox": "Medio-alto (prompt más largo que zero-shot)",
    },
}

df_comparacion = pd.DataFrame(resultados).set_index("metodo")
df_comparacion["tiempo_ms_por_frase"] = (pd.Series(tiempos) * 1000).round(2)
for columna in ["requiere_gpu", "explica_decision", "costo_aprox"]:
    df_comparacion[columna] = [metadatos_metodos[m][columna] for m in df_comparacion.index]

df_comparacion

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=df_comparacion.index, y=df_comparacion["f1_macro"], ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("F1 macro")
ax.set_title("Comparación de F1 macro por método")
plt.xticks(rotation=15, ha="right")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
metodos_para_graficar = {
    "Diccionario": df_eval["pred_diccionario"],
    "FinBERT": df_eval["pred_finbert"],
    "LLM zero-shot": df_eval["pred_llm_zero"],
    "LLM few-shot": df_eval["pred_llm_few"],
}
for ax, (nombre, predicciones) in zip(axes.ravel(), metodos_para_graficar.items()):
    graficar_matriz_confusion(df_eval["etiqueta"], predicciones, nombre, ax=ax)
plt.tight_layout()
plt.show()

### 6.1 Discusión: ¿cuándo conviene cada enfoque?

La siguiente celda genera un resumen a partir de **las métricas que efectivamente obtuviste** en esta
ejecución (van a variar según la muestra, la semilla y el modelo de LLM elegido, así que no asumimos de
antemano cuál "gana").

In [ ]:
mejor_metodo = df_comparacion["f1_macro"].idxmax()
metodo_mas_rapido = df_comparacion["tiempo_ms_por_frase"].idxmin()

resumen = (
    f"En esta ejecución (N_EVAL={len(df_eval)}, modelo LLM `{LLM_MODEL_NAME}`), el método con mayor "
    f"F1 macro fue **{mejor_metodo}** ({df_comparacion.loc[mejor_metodo, 'f1_macro']:.3f}), y el más "
    f"rápido por frase fue **{metodo_mas_rapido}** "
    f"({df_comparacion.loc[metodo_mas_rapido, 'tiempo_ms_por_frase']:.1f} ms/frase).\n\n"
    "Estos resultados dependen de la muestra, la semilla y el modelo elegido — vuelve a ejecutar esta "
    "celda después de cambiar `N_EVAL` o `LLM_MODEL_NAME` y compara."
)
display(Markdown(resumen))

Más allá del número que haya ganado hoy, en un contexto financiero real la elección no depende solo de
la métrica de exactitud:

| Criterio | Diccionario | FinBERT | LLM (zero/few-shot) |
|---|---|---|---|
| **Costo por frase** | Prácticamente nulo | Bajo (modelo propio, sin costo por llamada) | Medio-alto (más cómputo, o costo de API si usas un LLM externo) |
| **Velocidad** | Muy alta | Alta | Baja-media (generación token a token) |
| **Interpretabilidad** | Total (ves las palabras contadas) | Parcial (probabilidades, no razones) | Alta si el prompt pide justificación — pero la justificación puede ser post-hoc |
| **Necesita GPU** | No | No (ayuda, no es indispensable) | Prácticamente sí, para volúmenes grandes |
| **Se adapta a tareas nuevas sin reentrenar** | No (hay que rehacer las listas) | No (haría falta reafinar el modelo) | Sí (basta con cambiar el prompt) |
| **Funciona en otros idiomas** | Solo si tienes el diccionario en ese idioma | No (entrenado en inglés financiero) | Sí, en distintos grados según el modelo |

En la práctica, muchos equipos de datos financieros usan **más de uno a la vez**: un diccionario o
FinBERT para procesar el volumen completo de noticias todos los días a bajo costo, y un LLM reservado
para los casos ambiguos, para tareas nuevas que aún no justifican entrenar un modelo dedicado, o para
generar explicaciones en lenguaje natural que un analista humano pueda revisar rápidamente.

<a id="sec7"></a>
## 7. Ejercicios para los estudiantes

### Ejercicio 1 (introductorio) — Titulares en español

Prueba el LLM (parte 3) con 5-10 titulares **en español** de noticias financieras colombianas (puedes
buscarlos en La República, Portafolio o Valora Analitik). ¿El modelo sigue clasificando bien el
sentimiento aunque el prompt esté en inglés/español mezclado? ¿Cambia la calidad de la justificación?

*Pista: no necesitas tocar `PROMPT_SISTEMA` para esto — solo la lista de frases que le pasas a
`generar_respuestas`.*

### Ejercicio 2 (intermedio) — Clasificar también la temática

Modifica `PROMPT_SISTEMA` y `parsear_respuesta_llm` para que, además del sentimiento, el modelo
devuelva una tercera clave `"tematica"` con una de estas categorías: `"resultados"`, `"fusiones"`,
`"regulacion"`, `"macro"` u `"otro"`. Evalúa manualmente sobre 15-20 frases (no tenemos etiquetas reales
de temática en este dataset) qué tan razonables son las clasificaciones.

*Pista: agrega la nueva clave al JSON de ejemplo en el prompt, y actualiza el parser para leer y validar
también `"tematica"` contra la lista de categorías permitidas — igual que hacemos con `"sentimiento"`.*

### Ejercicio 3 (retador) — Conexión con el proyecto integrador

Para una empresa que sigas en tu proyecto integrador:

1. Consigue un pequeño conjunto de titulares o noticias diarias de esa empresa (pueden ser reales, de
   los últimos meses).
2. Calcula el sentimiento diario agregado (por ejemplo, el promedio de `+1`/`0`/`-1` según la
   predicción del método que prefieras de este cuaderno).
3. Descarga los retornos diarios de la acción con `yfinance` (como en el cuaderno de redes neuronales de
   la sesión anterior) y compara el sentimiento del día $t$ contra el retorno del día $t+1$.
4. ¿Hay alguna relación visible? ¿Un gráfico de dispersión sugiere algo, o el ruido domina? Sé honesto
   en tu conclusión — en la práctica, esta relación suele ser débil y ruidosa, y **eso también es un
   resultado válido** para reportar.